In [0]:
cards_path = "/Volumes/banking_card_project4_catalog/bronze/bank_volume/cards_data.csv"

df_cards = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(cards_path)
)

display(df_cards)

df_cards.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("banking_card_project4_catalog.bronze.cards")

In [0]:
users_path = "/Volumes/banking_card_project4_catalog/bronze/bank_volume/users_data.csv"

df_users = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(users_path)
)

display(df_users)

df_users.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("banking_card_project4_catalog.bronze.users")

In [0]:
transactions_path = "/Volumes/banking_card_project4_catalog/bronze/bank_volume/transactions_data.csv"

df_transactions = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(transactions_path)
)

display(df_transactions.limit(10))

df_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("banking_card_project4_catalog.bronze.transactions")

In [0]:
from pyspark.sql import functions as F

# ==========================================
# READ MCC JSON
# ==========================================

mcc_path = "/Volumes/banking_card_project4_catalog/bronze/bank_volume/mcc_codes.json"

mcc_raw_df = (
    spark.read
    .option("multiLine", "true")
    .json(mcc_path)
)

# ==========================================
# CONVERT JSON MAP INTO TWO COLUMNS
# ==========================================

mcc_bronze_df = (
    mcc_raw_df
    .select(
        F.explode(
            F.map_entries(
                F.create_map(
                    *[
                        x
                        for c in mcc_raw_df.columns
                        for x in (F.lit(c), F.col(f"`{c}`"))
                    ]
                )
            )
        ).alias("data")
    )
    .select(
        F.col("data.key").cast("int").alias("mcc"),
        F.col("data.value").cast("string").alias("category")
    )
)

# Check result
mcc_bronze_df.show(10, truncate=False)

mcc_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("banking_card_project4_catalog.bronze.mcc_codes")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    MapType,
    StringType
)

# ==========================================
# JSON FILE PATH
# ==========================================

fraud_path = "/Volumes/banking_card_project4_catalog/bronze/bank_volume/train_fraud_labels.json"


# ==========================================
# READ JSON AS RAW TEXT
# DO NOT USE spark.read.json()
# ==========================================

df_raw = (
    spark.read
    .text(fraud_path)
)


# ==========================================
# DEFINE JSON STRUCTURE
# target = MAP
# transaction_id -> fraud_label
# ==========================================

fraud_schema = StructType([
    StructField(
        "target",
        MapType(
            StringType(),
            StringType()
        ),
        True
    )
])


# ==========================================
# PARSE RAW JSON
# ==========================================

df_fraud_parsed = (
    df_raw
    .select(
        F.from_json(
            F.col("value"),
            fraud_schema
        ).alias("data")
    )
)


# ==========================================
# EXTRACT TARGET MAP
# ==========================================

df_fraud_map = (
    df_fraud_parsed
    .select(
        F.col("data.target").alias("target")
    )
)


# ==========================================
# FLATTEN JSON
# ==========================================

df_fraud_final = (
    df_fraud_map
    .select(
        F.explode(
            F.col("target")
        ).alias(
            "transaction_id",
            "fraud_label"
        )
    )
    .withColumn(
        "transaction_id",
        F.col("transaction_id").cast("long")
    )
)


# ==========================================
# WRITE TO DELTA TABLE
# ==========================================

(
    df_fraud_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "banking_card_project4_catalog.bronze.train_fraud_labels"
    )
)

In [0]:
%sql
USE CATALOG `banking_card_project4_catalog`;
USE SCHEMA bronze;
select * from bronze.mcc_codes